# HGND Reconstruction GNN — Results Analysis (SMASH)

This notebook loads GNN prediction results, merges them with the original SMASH simulation data, and reproduces all key performance plots:
- ROC curves (hit, edge, cluster levels)
- Score distributions (signal vs background)
- Efficiency / purity curves
- Energy resolution and linearity
- Per-event neutron detection metrics
- Neutron multiplicity confusion matrices

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import LogNorm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams.update({'font.size': 14})

from sklearn.metrics import (
    roc_curve, roc_auc_score, RocCurveDisplay,
    precision_recall_curve, auc,
)

# ── Paths ─────────────────────────────────────────────────────────────────
PACKAGE_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PACKAGE_ROOT not in sys.path:
    sys.path.insert(0, PACKAGE_ROOT)

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RESULTS_DIR  = os.path.join(os.getcwd(), 'results')
PLOTS_DIR    = os.path.join(PROJECT_ROOT, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f'Results dir: {RESULTS_DIR}')
print(f'Plots dir:   {PLOTS_DIR}')

## 1. Load Prediction DataFrames

In [ ]:
# Load the three prediction pickle files
clhitdf = pd.read_pickle(os.path.join(RESULTS_DIR, 'pred_hits_smash.pkl'))
cldf    = pd.read_pickle(os.path.join(RESULTS_DIR, 'pred_clusters_smash.pkl'))
edgesdf = pd.read_pickle(os.path.join(RESULTS_DIR, 'pred_edges_smash.pkl'))

print(f'Hit predictions:     {clhitdf.shape}')
print(f'Cluster predictions: {cldf.shape}')
print(f'Edge predictions:    {edgesdf.shape}')
print(f'\nclhitdf columns: {list(clhitdf.columns)}')
print(f'cldf columns:    {list(cldf.columns)}')
print(f'edgesdf columns: {list(edgesdf.columns)}')
clhitdf.head()

## 2. Load Original CSV Data and Merge with Predictions

Load the original SMASH hit DataFrame (via `load_hits()`) and merge with prediction DataFrames using `(Row, Instance)` and `(Row, ClusterID)` keys.

In [ ]:
from HGNDRecoGNN.data.graph_dataset import load_hits, prepare_halves, FEATURES

HITS_CSV_DIR = os.path.join(PROJECT_ROOT, 'data', 'smash_xecs_2.87gev_hardSkyrme_defaultSpot')

# Load the full DataFrame (uses parquet cache if available)
df = load_hits(HITS_CSV_DIR, cache_dir=os.path.join(os.getcwd(), 'cache', 'ndet_dataset_smash_defaultSpot', 'processed'))

print(f'Original DataFrame: {len(df)} rows, {df.Row.nunique()} events')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# ── Compute derived columns on the original DataFrame ─────────────────────
# Nn: number of unique neutron tracks per event
df['Nn'] = df[(df.n0_label == 1) & (df.Ekin > 0.)].groupby('Row')['Id'].transform('nunique')
df['Nn'] = df.groupby('Row')['Nn'].transform('max')
df['Nn'] = df['Nn'].fillna(0).astype(int)

# Nprompts: number of prompt hits per neutron track  
df['Nprompts'] = df[(df.n0_label == 1) & (df.Ekin > 0.)].groupby(['Row', 'Id'])['Instance'].transform('nunique')
df['Nprompts'] = df['Nprompts'].fillna(0).astype(int)

# Npn: number of primary neutrons (fMotherId == -1) per event
df['Npn'] = df[(df.n0_label == 1) & (df.fMotherId == -1)].groupby('Row')['Id'].transform('nunique')
df['Npn'] = df.groupby('Row')['Npn'].transform('max')
df['Npn'] = df['Npn'].fillna(0).astype(int)

# vNn: visual neutron count (n0_label==1 & Nprompts > 1)
df['vNn'] = df[(df.n0_label == 1) & (df.Ekin > 0.) & (df.Nprompts > 1)].groupby('Row')['Id'].transform('nunique')
df['vNn'] = df.groupby('Row')['vNn'].transform('max')
df['vNn'] = df['vNn'].fillna(0).astype(int)

print(f'Nn distribution:')
print(df.groupby('Row')['Nn'].max().value_counts().sort_index())

In [ ]:
# ── Merge predictions with original DataFrame ─────────────────────────────
# ClusterID is 0-indexed per graph (per half-event), so (Row, ClusterID) alone
# is NOT unique — a Row may appear as both top (istop=1) and bottom (istop=0)
# with overlapping ClusterIDs.  The correct join key is (Row, ClusterID, istop).

# Create the Instance mapping: within each (Row, istop), rank hits 0-based
df_top = df[df.fY > 0].copy()
df_bot = df[df.fY < 0].copy()

# Drop PDG==0 rows (same filtering as prepare_halves)
for half in [df_top, df_bot]:
    to_drop = half[half.PDG == 0].Row.unique()
    half.drop(half[half.Row.isin(to_drop)].index, inplace=True)

df_top['istop'] = 1
df_bot['istop'] = 0

# Assign Instance as 0-based within-event index (matches prediction ordering)
df_top['Instance_graph'] = df_top.groupby('Row').cumcount()
df_bot['Instance_graph'] = df_bot.groupby('Row').cumcount()

df_both = pd.concat([df_top, df_bot], ignore_index=True)

# Recompute Nn etc. on the filtered df — only for Rows that are in predictions
predicted_rows = set(clhitdf.Row.unique())
df_both_pred = df_both[df_both.Row.isin(predicted_rows)]

df_both['Nn'] = df_both_pred[(df_both_pred.n0_label == 1) & (df_both_pred.Ekin > 0.)].groupby('Row')['Id'].transform('nunique')
df_both['Nn'] = df_both.groupby('Row')['Nn'].transform('max').fillna(0).astype(int)
df_both['Nprompts'] = df_both_pred[(df_both_pred.n0_label == 1) & (df_both_pred.Ekin > 0.)].groupby(['Row', 'Id'])['Instance'].transform('nunique')
df_both['Nprompts'] = df_both['Nprompts'].fillna(0).astype(int)
df_both['Npn'] = df_both_pred[(df_both_pred.n0_label == 1) & (df_both_pred.fMotherId == -1)].groupby('Row')['Id'].transform('nunique')
df_both['Npn'] = df_both.groupby('Row')['Npn'].transform('max').fillna(0).astype(int)

print(f'df_both: {len(df_both)} rows, {df_both.Row.nunique()} events')
print(f'Predicted test hits: {len(clhitdf)} across {clhitdf.Row.nunique()} events (subset of test events)')

# ── Hit-level merge ────────────────────────────────────────────────────────
# Right-join: keep only events that were predicted
testdf = pd.merge(
    df_both, clhitdf,
    left_on=['Row', 'Instance_graph', 'istop'],
    right_on=['Row', 'Instance', 'istop'],
    how='right',
    suffixes=('_orig', ''),
)

# ── Cluster-level merge ────────────────────────────────────────────────────
# cldf has cl_istop column — use it as part of the join key to avoid duplicates
# when a Row appears in both top and bottom halves
cldf_keyed = cldf.copy()
cldf_keyed = cldf_keyed.rename(columns={'cl_istop': 'istop'})

testdf = pd.merge(
    testdf, cldf_keyed,
    on=['Row', 'ClusterID', 'istop'],
    how='left',
)

nan_score = testdf.cl_score.isna().sum()
print(f'\nMerged testdf: {len(testdf)} rows, {testdf.Row.nunique()} events')
print(f'Columns: {list(testdf.columns)}')
print(f'NaN in score: {testdf.score.isna().sum()}, NaN in cl_score: {nan_score}')
if nan_score > 0:
    print(f'  → {nan_score} hits have no cluster prediction (orphan hits); filling cl_score=0')
    testdf['cl_score'] = testdf['cl_score'].fillna(0.)
    testdf['cl_label'] = testdf['cl_label'].fillna(0).astype(int)
    testdf['e_pred']   = testdf['e_pred'].fillna(0.)
    testdf['e_true']   = testdf['e_true'].fillna(0.)
testdf.head(3)

In [ ]:
# ── Compute cluster-level derived columns ──────────────────────────────────
# Number of hits per cluster
testdf['Nclhits'] = testdf.groupby(['Row', 'ClusterID'])['score'].transform('count')

# Number of prompt (n0_label==1) hits per cluster
testdf['Nclhits_prompt'] = testdf[testdf.n0_label == 1].groupby(['Row', 'ClusterID'])['score'].transform('count')
testdf['Nclhits_prompt'] = testdf.groupby(['Row', 'ClusterID'])['Nclhits_prompt'].transform('max').fillna(0).astype(int)

# Fraction of prompt hits in cluster
testdf['Frclhits_prompt'] = testdf['Nclhits_prompt'] / testdf['Nclhits']

# Relative energy error
testdf['dErel'] = (testdf['e_pred'] - testdf['e_true']) / (testdf['e_true'] + 1e-9)

print(f'Nclhits range:  {testdf.Nclhits.min()} — {testdf.Nclhits.max()}')
print(f'e_true range:   {testdf.e_true.min():.3f} — {testdf.e_true.max():.3f}')
print(f'e_pred range:   {testdf.e_pred.min():.3f} — {testdf.e_pred.max():.3f}')

## 3. Hit-Level ROC Curve and AUC

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.gca()

fpr, tpr, _ = roc_curve(testdf.label, testdf.score)
hit_auc = roc_auc_score(testdf.label, testdf.score)

ax.plot(fpr, tpr, linewidth=2, label=f'Hit classifier (AUC = {hit_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Hit-level neutron classification')
ax.legend(loc=4)
ax.grid(True)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'hit_ROC.pdf'))
print(f'Hit AUC = {hit_auc:.4f}')

## 4. Edge-Level (Link) ROC Curve and AUC

In [ ]:
fig = plt.figure(figsize=(7, 6))
ax = fig.gca()

fpr_e, tpr_e, _ = roc_curve(edgesdf.link_label, edgesdf.link_score)
link_auc = roc_auc_score(edgesdf.link_label, edgesdf.link_score)

ax.plot(fpr_e, tpr_e, linewidth=2, label=f'Link classifier (AUC = {link_auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Edge-level link classification')
ax.legend(loc=4)
ax.grid(True)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'link_ROC.pdf'))
print(f'Link AUC = {link_auc:.4f}')

## 5. Cluster-Level Classification ROC Curve (top / bottom / combined)

In [ ]:
# Cluster-level: one score per cluster (take max over hits in each cluster)
cl_unique = testdf.groupby(['Row', 'ClusterID']).agg(
    cl_label=('cl_label', 'max'),
    cl_score=('cl_score', 'max'),
    istop=('istop', 'max'),
).reset_index()

fig = plt.figure(figsize=(7, 6))
ax = fig.gca()

# Combined
fpr_c, tpr_c, _ = roc_curve(cl_unique.cl_label, cl_unique.cl_score)
cl_auc = roc_auc_score(cl_unique.cl_label, cl_unique.cl_score)
ax.plot(fpr_c, tpr_c, linewidth=2, label=f'Combined (AUC = {cl_auc:.4f})')

# Top
top = cl_unique[cl_unique.istop == 1]
cl_auc_top = np.nan
if len(top) > 0 and top.cl_label.nunique() > 1:
    fpr_t, tpr_t, _ = roc_curve(top.cl_label, top.cl_score)
    cl_auc_top = roc_auc_score(top.cl_label, top.cl_score)
    ax.plot(fpr_t, tpr_t, linewidth=1.5, linestyle='--', label=f'Top (AUC = {cl_auc_top:.4f})')
else:
    print(f'Top half: {len(top)} clusters, {top.cl_label.nunique()} unique labels — skipping ROC')

# Bottom
bot = cl_unique[cl_unique.istop == 0]
cl_auc_bot = np.nan
if len(bot) > 0 and bot.cl_label.nunique() > 1:
    fpr_b, tpr_b, _ = roc_curve(bot.cl_label, bot.cl_score)
    cl_auc_bot = roc_auc_score(bot.cl_label, bot.cl_score)
    ax.plot(fpr_b, tpr_b, linewidth=1.5, linestyle=':', label=f'Bottom (AUC = {cl_auc_bot:.4f})')
else:
    print(f'Bottom half: {len(bot)} clusters, {bot.cl_label.nunique() if len(bot) > 0 else 0} unique labels — skipping ROC')

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('Cluster classification performance')
ax.legend(loc=4)
ax.grid(True)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'cl_ROC.pdf'))

print(f'\nistop distribution in test set: {testdf.istop.value_counts().to_dict()}')

## 6. Hit Score Distribution (Signal vs Background)

In [ ]:
fig = plt.figure(figsize=(10, 6))
plt.hist(testdf[testdf.label == 0].score, bins=100, range=[0, 1],
         density=True, alpha=0.5, label='Background', color='grey')
plt.hist(testdf[testdf.label == 1].score, bins=100, range=[0, 1],
         density=True, alpha=0.7, histtype='step', linewidth=2,
         label='Signal ($n^0$ hits)', color='green')
plt.axvline(0.5, color='red', linestyle='--', alpha=0.5, label='Threshold = 0.5')
plt.xlabel('Hit score')
plt.ylabel('Normalised entries')
plt.title('Hit-level score distribution')
plt.legend()
plt.yscale('log')
plt.grid(None)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'hit_score_dist.pdf'))

## 7. Cluster Score Distribution (Signal vs Background, Top vs Bottom)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, half) in zip(axes, [('Top', cl_unique[cl_unique.istop == 1]),
                                     ('Bottom', cl_unique[cl_unique.istop == 0])]):
    ax.hist(half[half.cl_label == 0].cl_score, bins=50, range=[0, 1],
            density=True, alpha=0.5, label='Background', color='grey')
    ax.hist(half[half.cl_label == 1].cl_score, bins=50, range=[0, 1],
            density=True, alpha=0.7, histtype='step', linewidth=2,
            label='Signal', color='green')
    ax.set_xlabel('Cluster score')
    ax.set_ylabel('Normalised entries')
    ax.set_title(f'Cluster score — {label}')
    ax.legend()
    ax.set_yscale('log')

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'cl_score_dist.pdf'))

## 8. Link Score Distribution

In [ ]:
fig = plt.figure(figsize=(10, 6))
plt.hist(edgesdf[edgesdf.link_label == 0].link_score, bins=100, range=[0, 1],
         density=True, alpha=0.5, label='False links', color='grey')
plt.hist(edgesdf[edgesdf.link_label == 1].link_score, bins=100, range=[0, 1],
         density=True, alpha=0.7, histtype='step', linewidth=2,
         label='True links', color='blue')
plt.xlabel('Link score')
plt.ylabel('Normalised entries')
plt.title('Edge-level link score distribution')
plt.legend()
plt.yscale('log')
plt.grid(None)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'link_score_dist.pdf'))

## 9. Neutron Hit Efficiency vs Purity Curves

In [ ]:
# ── Cluster-level efficiency & purity vs threshold ────────────────────────
# Use the best cluster per event (highest cl_score)
resdf = testdf.sort_values('cl_score', ascending=False).groupby('Row').head(1).copy()

ts, purs, effs = [], [], []
for thres in np.arange(0, 0.95, 0.025):
    sel = resdf.cl_score > thres
    n_selected = sel.sum()
    if n_selected == 0:
        ts.append(thres); purs.append(0); effs.append(0)
        continue
    ts.append(thres)
    purs.append(len(resdf[sel & (resdf.Nn > 0)]) / max(n_selected, 1))
    effs.append(len(resdf[sel & (resdf.Nn > 0)]) / max(resdf.Nn.sum(), 1))

fig = plt.figure(figsize=(10, 6))
plt.plot(ts, purs, linewidth=2, label='Purity')
plt.plot(ts, effs, linewidth=2, label='Efficiency')
plt.ylim(0, 1.1)
plt.xlabel('Threshold', fontsize=16)
plt.ylabel('Metric value')
plt.title('Single best cluster per event')
plt.legend(loc=2, fontsize=14)
plt.grid(True)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'effpur_singlecl.pdf'))

In [ ]:
# ── Energy spectra with efficiency/purity ratio panel ─────────────────────
plt.rcParams.update({'font.size': 18})

for thres in [0., 0.1, 0.2, 0.3, 0.5, 0.7]:
    fig, (ax_main, ax_ratio) = plt.subplots(
        nrows=2, ncols=1, figsize=(10, 10), sharex=True,
        gridspec_kw={'height_ratios': [3, 1]}
    )
    bins = 40
    sel = resdf.cl_score > thres

    counts0, bins0, _ = ax_main.hist(
        testdf[testdf.n0_label == 1].groupby(['Row', 'Id']).Ekin.apply('mean'),
        log=False, bins=bins, range=[0, 6], alpha=0.5,
        label='all signal neutrons'
    )
    counts1, bins1, _ = ax_main.hist(
        resdf[sel & (resdf.e_true > 0)].e_true,
        linewidth=4, bins=bins, log=False, range=[0, 6],
        histtype='step', label='$E_{true}$'
    )
    counts2, bins2, _ = ax_main.hist(
        resdf[sel].e_pred,
        linewidth=4, bins=bins, log=False, range=[0, 6],
        histtype='step', label='$E_{predicted}$'
    )
    counts3, bins3, _ = ax_main.hist(
        resdf[sel & (resdf.cl_label == 0)].e_pred,
        linewidth=4, linestyle='dashed', bins=bins, log=False, range=[0, 6],
        histtype='step', label='$E_{predicted}$ for fake'
    )

    eff = counts1 / (counts0 + 1e-9)
    pur = 1 - counts3 / (counts2 + 1e-9)

    ax_main.set_ylabel('N events')
    ax_main.set_title(f'Best cluster per event. Threshold = {thres}')
    if thres < 0.1:
        ax_main.legend(loc=1, title=f'Threshold = {thres}')

    bin_centers = bins0[:-1] + np.diff(bins0) / 2
    ax_ratio.plot(bin_centers, eff, ':', linewidth=4, color='blue', label='~efficiency')
    ax_ratio.plot(bin_centers, pur, ':', linewidth=4, color='red', label='~purity')
    ax_ratio.legend(ncols=2)
    ax_ratio.grid()
    ax_ratio.set_ylim(0, 1.4)
    ax_ratio.set_xlabel('$E_{kin}\\ [GeV]$')

    plt.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, f'spectra_bestcl_{thres}.pdf'))
    plt.show()

plt.rcParams.update({'font.size': 14})

## 10. Cluster Energy Resolution: Predicted vs True Energy

In [ ]:
# ── E_reco vs E_true 2D histogram (cluster level) ─────────────────────────
fig = plt.figure(figsize=(7, 6))
sel = testdf.cl_score > 0.
e_true_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max')
e_pred_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_pred.apply('max')

plt.hist2d(e_true_cl, e_pred_cl,
           range=[[0.1, 5], [0, 5]], bins=[100, 100], norm=LogNorm())
plt.plot([0, 5], [0, 5], 'r--', linewidth=1, alpha=0.5)
plt.title('$Cluster\\ level$')
plt.xlabel('$E_{true},\\ GeV$')
plt.ylabel('$E_{reco},\\ GeV$')
plt.colorbar()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'ereco.pdf'))

In [ ]:
# ── E_reco vs E_true at multiple thresholds ───────────────────────────────
for threshold in [0., 0.1, 0.2, 0.3, 0.5, 0.7]:
    fig = plt.figure(figsize=(7, 6))
    sel = testdf.cl_score > threshold
    e_true_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max')
    e_pred_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_pred.apply('max')
    plt.hist2d(e_true_cl, e_pred_cl,
               range=[[0, 5], [0, 5]], bins=[100, 100], norm=LogNorm())
    plt.plot([0, 5], [0, 5], 'r--', linewidth=1, alpha=0.5)
    plt.title(f'$Cluster\\ level.\\ Threshold = ${threshold}')
    plt.xlabel('$E_{true},\\ GeV$')
    plt.ylabel('$E_{reco},\\ GeV$')
    plt.grid(True, alpha=0.3)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Relative energy resolution distribution ───────────────────────────────
fig = plt.figure(figsize=(10, 8))
for threshold in [0., 0.1, 0.2, 0.5, 0.7]:
    sel = (testdf.cl_score >= threshold) & (testdf.e_true > 0.5)
    errs = testdf[sel].groupby(['Row', 'ClusterID'])['dErel'].apply('mean')
    plt.hist(errs, bins=100, range=[-1, 1], density=True,
             label=f'threshold = {threshold}\n std = {errs.std():.3f}'
                   f'\n mean = {errs.mean():.3f}')
plt.legend(fontsize=12)
plt.xlabel('$\\dfrac{E_{pred} - E_{true}}{E_{true}}$', fontsize=16)
plt.ylabel('Normalised entries')
plt.title('Relative energy resolution (cluster level)')
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'rel_err.pdf'))

## 11. Energy Resolution as a Function of True Energy

In [ ]:
# ── Energy linearity: (E_pred - E_true)/E_true vs E_true ──────────────────
fig = plt.figure(figsize=(10, 4))
sel = (testdf.cl_score > 0.2) & (testdf.cl_label == 1)
sns.regplot(
    x=testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max'),
    y=(testdf[sel].groupby(['Row', 'ClusterID']).e_pred.apply('max')
       - testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max'))
      / testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max'),
    x_bins=np.arange(0.1, 5., 0.2), order=1, fit_reg=False,
    scatter_kws={'lw': 0.05, 'marker': '+', 'color': 'black'},
    ci=95,
)
plt.xlabel('$E_{true},\\ GeV$', fontsize=16)
plt.fill_between(np.arange(0., 5.1, 0.2), -0.1, 0.1, alpha=0.1)
plt.grid(True, alpha=0.3)
plt.xlim(0, 5)
plt.ylim(-0.8, 0.8)
plt.ylabel('$\\dfrac{E_{pred} - E_{true}}{E_{true}}$', fontsize=16)
plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'linearity.pdf'))

In [ ]:
# ── Resolution (sigma) vs E_true, split by top/bottom ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, istop_val) in zip(axes, [('Top', 1), ('Bottom', 0)]):
    for threshold in [0.2, 0.5, 0.7]:
        sel = (testdf.cl_score > threshold) & (testdf.cl_label == 1) & (testdf.istop == istop_val)
        if sel.sum() == 0:
            continue
        ebins = np.arange(0.2, 5.1, 0.3)
        testdf_sel = testdf[sel].copy()
        testdf_sel['Ebin'] = pd.cut(testdf_sel['e_true'], bins=ebins)
        binned_std = testdf_sel.groupby('Ebin')['dErel'].std()
        binned_mean = testdf_sel.groupby('Ebin')['dErel'].mean()
        bin_centers = [(b.left + b.right) / 2 for b in binned_std.index]
        
        ax.errorbar(bin_centers, binned_mean, yerr=binned_std,
                     fmt='o-', capsize=3, linewidth=1.5,
                     label=f'thr={threshold} (σ={binned_std.mean():.3f})')
    
    ax.set_xlabel('$E_{true},\\ GeV$', fontsize=14)
    ax.set_ylabel('$\\langle dE_{rel} \\rangle \\pm \\sigma$', fontsize=14)
    ax.set_title(f'{label} detector half')
    ax.legend(fontsize=10)
    ax.set_xlim(0, 5)
    ax.set_ylim(-0.5, 0.5)
    ax.axhline(0, color='grey', linestyle='--', alpha=0.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'resolution_vs_etrue.pdf'))

## 12. Per-Event Neutron Detection Efficiency

In [ ]:
# ── Score vs cluster properties (2D histograms) ──────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Score vs N_cluster_hits
ax = axes[0]
im = ax.hist2d(testdf.groupby(['Row', 'ClusterID']).cl_score.apply('max'),
               testdf.groupby(['Row', 'ClusterID']).Nclhits.apply('max'),
               range=[[0, 1], [0, 50]], bins=[50, 50], norm=LogNorm())
ax.set_xlabel('Cluster score')
ax.set_ylabel('$N_{cluster\\ hits}$')
ax.set_title('Score vs cluster hits')
plt.colorbar(im[3], ax=ax)

# Score vs N_prompt_hits
ax = axes[1]
im = ax.hist2d(testdf.groupby(['Row', 'ClusterID']).cl_score.apply('max'),
               testdf.groupby(['Row', 'ClusterID']).Nclhits_prompt.apply('max'),
               range=[[0, 1], [0, 30]], bins=[50, 30], norm=LogNorm())
ax.set_xlabel('Cluster score')
ax.set_ylabel('$N_{prompt\\ hits}$')
ax.set_title('Score vs prompt hits')
plt.colorbar(im[3], ax=ax)

# Score vs E_true
ax = axes[2]
im = ax.hist2d(testdf.groupby(['Row', 'ClusterID']).cl_score.apply('max'),
               testdf.groupby(['Row', 'ClusterID']).e_true.apply('max'),
               range=[[0, 1], [0, 5]], bins=[50, 50], norm=LogNorm())
ax.set_xlabel('Cluster score')
ax.set_ylabel('$E_{true},\\ GeV$')
ax.set_title('Score vs true energy')
plt.colorbar(im[3], ax=ax)

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'score_vs_properties.pdf'))

In [ ]:
# ── Neutron multiplicity confusion matrix (N_reco vs N_true) ──────────────
plt.rcParams.update({'font.size': 14})

for threshold in [0.1, 0.2, 0.3, 0.5, 0.7]:
    sel = (testdf.cl_score >= threshold) & (testdf.e_pred > 0.)
    nreco = testdf[sel].groupby('Row')['ClusterID'].apply('nunique')
    
    # Add zero counts for events with no cluster above threshold
    rows_present = nreco.index
    sel_zero = ~testdf.Row.isin(rows_present)
    zerocounts = testdf[sel_zero].groupby('Row')['ClusterID'].apply('nunique') * 0
    nreco = pd.concat([nreco, zerocounts])
    
    ntrue = testdf.groupby('Row')['Nn'].apply('max')
    
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    hist, xbins, ybins, im = ax.hist2d(
        np.clip(nreco, 0, 3),
        np.clip(ntrue[nreco.index], 0, 3),
        range=[[-0.5, 3.5], [-0.5, 3.5]], bins=[4, 4],
        norm=LogNorm(vmax=len(testdf.Row.unique())),
        density=False,
    )
    
    for i in range(len(ybins) - 1):
        for j in range(len(xbins) - 1):
            color = 'black' if hist.T[i, j] >= np.quantile(hist, 0.95) else 'white'
            ax.text(xbins[j] + 0.6, ybins[i] + 0.5, int(np.round(hist.T[i, j], 0)),
                    ha='center', va='center', c=color)
            col_sum = np.sum(hist, axis=0)[j]
            if col_sum > 0:
                ax.text(xbins[j] + 0.65, ybins[i] + 0.15,
                        np.round(hist.T[i, j] / col_sum, 3),
                        ha='center', va='center', fontweight='bold', c=color)
    
    ax.set_title(f'threshold = {threshold}')
    ax.set_ylabel('$N_{true}$', fontsize=16)
    ax.set_yticks(np.arange(4))
    ax.set_xlabel('$N_{reco}$', fontsize=16)
    ax.set_xticks(np.arange(4))
    plt.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, f'confusion_4x4_thr{threshold}.pdf'))
    plt.show()

In [ ]:
# ── Neutron multiplicity distribution ─────────────────────────────────────
ran = [-0.5, 4.5]
bins = 5
plt.rcParams.update({'font.size': 18})

fig = plt.figure(figsize=(8, 6))
ax = plt.subplot(111)
ax.hist(testdf.groupby('Row').Nn.apply('max'), density=True,
        color='green', linewidth=2, histtype='stepfilled',
        label='$n^0$', alpha=0.8, range=ran, bins=bins)
ax.hist(testdf.groupby('Row').Npn.apply('max'), density=True,
        color='black', linewidth=2, histtype='step',
        label='${n^0}_{prim}$', alpha=0.8, range=ran, bins=bins)
ax.set_xlim(-0.5, 4.5)
ax.legend()
ax.set_xlabel('$N_{particles}$')
ax.set_ylabel('Normalised entries')
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR, 'nparticles.pdf'))
plt.show()
plt.rcParams.update({'font.size': 14})

## 13. Performance Comparison: Top vs Bottom Detector Halves

In [ ]:
# ── Side-by-side comparison: Top vs Bottom ────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Row 1: Hit ROC
for i, (label, istop_val) in enumerate([('Top', 1), ('Bottom', 0)]):
    ax = axes[0, i]
    sub = testdf[testdf.istop == istop_val]
    if len(sub) > 0 and sub.label.nunique() >= 2:
        fpr_h, tpr_h, _ = roc_curve(sub.label, sub.score)
        auc_h = roc_auc_score(sub.label, sub.score)
        ax.plot(fpr_h, tpr_h, linewidth=2, label=f'Hit AUC = {auc_h:.4f}')
        
        # Cluster ROC on same axes
        sub_cl = cl_unique[cl_unique.istop == istop_val]
        if len(sub_cl) > 0 and sub_cl.cl_label.nunique() >= 2:
            fpr_cl, tpr_cl, _ = roc_curve(sub_cl.cl_label, sub_cl.cl_score)
            auc_cl = roc_auc_score(sub_cl.cl_label, sub_cl.cl_score)
            ax.plot(fpr_cl, tpr_cl, linewidth=2, linestyle='--', label=f'Cluster AUC = {auc_cl:.4f}')
        
        ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        ax.legend(loc=4, fontsize=10)
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14, color='gray',
                transform=ax.transAxes)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(f'{label} — ROC curves')
    ax.grid(True)

# Row 2: E_reco vs E_true
for i, (label, istop_val) in enumerate([('Top', 1), ('Bottom', 0)]):
    ax = axes[1, i]
    sel = (testdf.cl_score > 0.2) & (testdf.cl_label == 1) & (testdf.istop == istop_val)
    e_true_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_true.apply('max')
    e_pred_cl = testdf[sel].groupby(['Row', 'ClusterID']).e_pred.apply('max')
    if len(e_true_cl) > 0:
        ax.hist2d(e_true_cl, e_pred_cl,
                  range=[[0.1, 5], [0, 5]], bins=[60, 60], norm=LogNorm())
        ax.plot([0, 5], [0, 5], 'r--', linewidth=1, alpha=0.5)
        dErel = (e_pred_cl - e_true_cl) / e_true_cl
        ax.set_title(f'{label} — E_reco vs E_true (σ={dErel.std():.3f})')
    else:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', fontsize=14, color='gray',
                transform=ax.transAxes)
        ax.set_title(f'{label} — E_reco vs E_true')
    ax.set_xlabel('$E_{true},\\ GeV$')
    ax.set_ylabel('$E_{reco},\\ GeV$')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
fig.savefig(os.path.join(PLOTS_DIR, 'topbot_comparison.pdf'))

In [ ]:
# ── XY hit maps (front wall) ──────────────────────────────────────────────
# Requires fX, fY columns from original data merge
if 'fX' in testdf.columns and 'fY' in testdf.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # All signal neutrons
    sel_sig = (testdf.cl_label == 1) & (testdf.istop == 1)
    groups_all = testdf[sel_sig].sort_values('fTime', ascending=False).groupby(['Row', 'ClusterID']).head(1)
    hist1, xe1, ye1 = np.histogram2d(groups_all.fX, groups_all.fY, bins=11)

    # High-score signal
    sel_hi = (testdf.cl_score > 0.3) & (testdf.cl_label == 1) & (testdf.istop == 1)
    groups_hi = testdf[sel_hi].sort_values('fTime', ascending=False).groupby(['Row', 'ClusterID']).head(1)
    hist2, xe2, ye2 = np.histogram2d(groups_hi.fX, groups_hi.fY, bins=11)

    hist_norm = hist2 / (hist1 + 1e-9)

    im1 = axes[0].imshow(hist1.T, origin='lower', extent=[xe1[0], xe1[-1], ye1[0], ye1[-1]])
    axes[0].set_title('Signal neutrons')
    axes[0].set_xlabel('$x,\\ cm$'); axes[0].set_ylabel('$y,\\ cm$')
    plt.colorbar(im1, ax=axes[0])

    im2 = axes[1].imshow(hist2.T, origin='lower', extent=[xe2[0], xe2[-1], ye2[0], ye2[-1]])
    axes[1].set_title('Cluster score $> 0.3$')
    axes[1].set_xlabel('$x,\\ cm$'); axes[1].set_ylabel('$y,\\ cm$')
    plt.colorbar(im2, ax=axes[1])

    im3 = axes[2].imshow(hist_norm.T, origin='lower', extent=[xe1[0], xe1[-1], ye1[0], ye1[-1]],
                          vmin=0.3, vmax=0.9)
    axes[2].set_title('Efficiency (normalised)')
    axes[2].set_xlabel('$x,\\ cm$'); axes[2].set_ylabel('$y,\\ cm$')
    plt.colorbar(im3, ax=axes[2])

    plt.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, 'n0frontwall_h03.pdf'))
    plt.show()
else:
    print('fX/fY columns not available — skipping XY hit maps')

In [ ]:
# ── eToF distribution: signal vs background ───────────────────────────────
if 'eToF' in testdf.columns:
    fig = plt.figure(figsize=(10, 6))
    r = [0, 10]; b = 50
    testdf[testdf.n0_label == 0].eToF.hist(range=r, bins=b,
                                           color='grey', alpha=0.5, label='background')
    testdf[testdf.n0_label == 1].eToF.hist(range=r, bins=b,
                                           linewidth=2, color='green', histtype='step',
                                           label='$n^0$ prompt hits')
    plt.xlabel('$E_{ToF\\ hit},\\ GeV$')
    plt.ylabel('entries')
    plt.legend(loc='best')
    plt.yscale('log')
    plt.grid(None)
    plt.tight_layout()
    fig.savefig(os.path.join(PLOTS_DIR, 'etofs.pdf'))
    plt.show()
else:
    print('eToF column not available — skipping')

## 14. Summary Performance Table

In [ ]:
# ── Summary performance table ─────────────────────────────────────────────
# Energy resolution for signal clusters (cl_label==1, cl_score>0.2)
sel_res = (testdf.cl_score > 0.2) & (testdf.cl_label == 1)
dErel_all = testdf[sel_res].groupby(['Row', 'ClusterID'])['dErel'].mean()
dErel_top = testdf[sel_res & (testdf.istop == 1)].groupby(['Row', 'ClusterID'])['dErel'].mean()
dErel_bot = testdf[sel_res & (testdf.istop == 0)].groupby(['Row', 'ClusterID'])['dErel'].mean()

# Per-event detection efficiency (at least one cluster with cl_score>0.3 & cl_label==1)
thres_eff = 0.3
n_total_events = testdf.Row.nunique()
events_with_neutron = testdf[testdf.Nn > 0].Row.nunique()
events_detected = testdf[(testdf.cl_score > thres_eff) & (testdf.cl_label == 1)].Row.nunique()
detection_eff = events_detected / max(events_with_neutron, 1)

summary = pd.DataFrame({
    'Metric': [
        'Hit AUC', 'Link AUC',
        'Cluster AUC (combined)', 'Cluster AUC (top)', 'Cluster AUC (bottom)',
        'Energy bias (mean dErel)', 'Energy resolution (σ dErel)',
        'Energy bias top', 'Energy resolution top',
        'Energy bias bottom', 'Energy resolution bottom',
        'Per-event detection eff (thr=0.3)',
        'Total test events', 'Events with neutrons',
    ],
    'Value': [
        f'{hit_auc:.4f}', f'{link_auc:.4f}',
        f'{cl_auc:.4f}', f'{cl_auc_top:.4f}', f'{cl_auc_bot:.4f}',
        f'{dErel_all.mean():.4f}', f'{dErel_all.std():.4f}',
        f'{dErel_top.mean():.4f}', f'{dErel_top.std():.4f}',
        f'{dErel_bot.mean():.4f}', f'{dErel_bot.std():.4f}',
        f'{detection_eff:.4f}',
        f'{n_total_events}', f'{events_with_neutron}',
    ],
})

# Save
summary.to_csv(os.path.join(RESULTS_DIR, 'summary_metrics_smash.csv'), index=False)
print(summary.to_latex(index=False))
summary.style.set_caption('GNN Performance Summary (SMASH)')

In [ ]:
# ── Diagnostic: check n0_label vs eToF, Ekin, PDG ────────────────────────
sig = testdf[testdf.n0_label == 1]
bkg = testdf[testdf.n0_label == 0]

print(f"Signal hits: {len(sig)},  Background hits: {len(bkg)}")
print(f"\n=== Signal hits (n0_label==1) ===")
print(f"  PDG values: {sig.PDG.value_counts().to_dict()}")
print(f"  Ekin:  min={sig.Ekin.min():.4f}  median={sig.Ekin.median():.4f}  max={sig.Ekin.max():.4f}")
print(f"  eToF:  min={sig.eToF.min():.4f}  median={sig.eToF.median():.4f}  max={sig.eToF.max():.4f}")
print(f"  Ekin < 0.1 GeV: {(sig.Ekin < 0.1).sum()}  ({(sig.Ekin < 0.1).mean()*100:.1f}%)")
print(f"  eToF < 0.2 GeV: {(sig.eToF < 0.2).sum()}  ({(sig.eToF < 0.2).mean()*100:.1f}%)")
print(f"  eToF < 0.5 GeV: {(sig.eToF < 0.5).sum()}  ({(sig.eToF < 0.5).mean()*100:.1f}%)")

# What does the low-eToF signal look like?
low_etof_sig = sig[sig.eToF < 0.5]
print(f"\n=== Signal hits with eToF < 0.5 GeV ({len(low_etof_sig)} hits) ===")
print(f"  Ekin distribution:")
print(low_etof_sig.Ekin.describe())
print(f"  fTime distribution:")
print(low_etof_sig.fTime.describe())

# Compare eToF to Ekin for signal
print(f"\n=== eToF vs Ekin consistency for signal ===")
print(f"  |eToF - Ekin| < 0.5: {((sig.eToF - sig.Ekin).abs() < 0.5).sum()} / {len(sig)}")
print(f"  |eToF - Ekin| > 2.0: {((sig.eToF - sig.Ekin).abs() > 2.0).sum()} / {len(sig)}")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. eToF for signal vs background
axes[0,0].hist(bkg.eToF, range=[0,10], bins=50, color='grey', alpha=0.5, label='bkg')
axes[0,0].hist(sig.eToF, range=[0,10], bins=50, color='green', histtype='step', lw=2, label='signal')
axes[0,0].set_xlabel('eToF [GeV]'); axes[0,0].set_ylabel('hits'); axes[0,0].set_yscale('log')
axes[0,0].legend(); axes[0,0].set_title('eToF: signal vs bkg')

# 2. Ekin for signal
axes[0,1].hist(sig.Ekin, range=[0,10], bins=50, color='red', histtype='step', lw=2, label='Ekin signal')
axes[0,1].set_xlabel('Ekin [GeV]'); axes[0,1].set_ylabel('hits'); axes[0,1].set_yscale('log')
axes[0,1].legend(); axes[0,1].set_title('MC Ekin of signal hits')

# 3. eToF vs Ekin scatter for signal
axes[0,2].hist2d(sig.Ekin, sig.eToF, range=[[0,5],[0,5]], bins=80, norm=LogNorm())
axes[0,2].plot([0,5],[0,5],'r--',lw=1)
axes[0,2].set_xlabel('Ekin [GeV]'); axes[0,2].set_ylabel('eToF [GeV]')
axes[0,2].set_title('Signal: Ekin vs eToF')

# 4. fMotherId distribution for signal
print(f"\n=== Signal fMotherId ===")
print(sig.fMotherId.value_counts().head(10))
axes[1,0].hist(sig.fMotherId.clip(-2, 100), range=[-2,50], bins=52, color='blue', histtype='step', lw=2)
axes[1,0].set_xlabel('fMotherId'); axes[1,0].set_ylabel('hits'); axes[1,0].set_yscale('log')
axes[1,0].set_title('Signal: fMotherId')

# 5. label (hit classification) vs n0_label  
print(f"\n=== label (GNN hit pred target) vs n0_label ===")
print(pd.crosstab(testdf.n0_label, testdf.label, margins=True))

# 6. Check: are there secondary neutrons being labeled as signal?
print(f"\n=== Signal hits: primary (fMotherId==-1) vs secondary ===")
primary_sig = sig[sig.fMotherId == -1]
secondary_sig = sig[sig.fMotherId != -1]
print(f"  Primary neutron hits: {len(primary_sig)} ({len(primary_sig)/len(sig)*100:.1f}%)")
print(f"  Secondary neutron hits: {len(secondary_sig)} ({len(secondary_sig)/len(sig)*100:.1f}%)")
print(f"  Secondary Ekin: min={secondary_sig.Ekin.min():.4f}  median={secondary_sig.Ekin.median():.4f}")
print(f"  Primary Ekin:   min={primary_sig.Ekin.min():.4f}  median={primary_sig.Ekin.median():.4f}")

axes[1,1].hist(primary_sig.eToF, range=[0,10], bins=50, color='blue', histtype='step', lw=2, label=f'primary (N={len(primary_sig)})')
axes[1,1].hist(secondary_sig.eToF, range=[0,10], bins=50, color='red', histtype='step', lw=2, label=f'secondary (N={len(secondary_sig)})')
axes[1,1].set_xlabel('eToF [GeV]'); axes[1,1].set_ylabel('hits'); axes[1,1].set_yscale('log')
axes[1,1].legend(); axes[1,1].set_title('eToF: primary vs secondary n')

# 7. Check fTime distribution for low-eToF signal hits
axes[1,2].hist(sig.fTime, range=[0,100], bins=50, color='green', histtype='step', lw=2, label='all signal')
axes[1,2].hist(low_etof_sig.fTime, range=[0,100], bins=50, color='red', histtype='step', lw=2, label='eToF < 0.5')
axes[1,2].set_xlabel('fTime [ns]'); axes[1,2].set_ylabel('hits'); axes[1,2].set_yscale('log')
axes[1,2].legend(); axes[1,2].set_title('fTime of signal hits')

plt.tight_layout()
plt.show()

In [ ]:
# ── Deeper diagnostic: where does the low-eToF peak come from? ───────────
sig = testdf[testdf.n0_label == 1].copy()

# n0_label definition: PDG==2112 & Ekin within [eToF_dn_2sig, eToF_up_2sig] & Side in [5,6]
# eToF is computed from (distance, fTime) — the measured time-of-flight energy
# The low eToF means slow neutron arrival (large fTime) or short distance

# Question: is the low-eToF peak physical (truly slow neutrons) or labeling artifact?

# Check: what is eToF_up_2sig and eToF_dn_2sig for these hits?
print("=== Columns available with eToF windows ===")
etof_cols = [c for c in testdf.columns if 'eToF' in c or 'etof' in c.lower()]
print(etof_cols)

# Are eToF_up_2sig / eToF_dn_2sig in testdf?
if 'eToF_up_2sig' in testdf.columns:
    print(f"\neToF_up_2sig available")
    print(f"Signal: eToF_up_2sig range: {sig.eToF_up_2sig.min():.3f} - {sig.eToF_up_2sig.max():.3f}")
    print(f"Signal: eToF_dn_2sig range: {sig.eToF_dn_2sig.min():.3f} - {sig.eToF_dn_2sig.max():.3f}")
    
    # The 2sigma window is quite wide for late-time hits. Check:
    sig['etof_window'] = sig.eToF_up_2sig - sig.eToF_dn_2sig
    print(f"\nToF window width (eToF_up - eToF_dn):")
    print(sig.etof_window.describe())
    
    # Low eToF signal hits: are they in a very wide window?
    low = sig[sig.eToF < 0.5]
    print(f"\nLow eToF (<0.5) signal hits window width:")
    print(low.etof_window.describe())

# Key question: are these actually slow neutrons (Ekin is also low)?
print(f"\n=== For signal hits with eToF < 0.5 ===")
low_sig = sig[sig.eToF < 0.5]
print(f"Ekin < 0.1: {(low_sig.Ekin < 0.1).sum()}")
print(f"Ekin 0.1-0.3: {((low_sig.Ekin >= 0.1) & (low_sig.Ekin < 0.3)).sum()}")
print(f"Ekin 0.3-0.5: {((low_sig.Ekin >= 0.3) & (low_sig.Ekin < 0.5)).sum()}")
print(f"Ekin > 0.5: {(low_sig.Ekin > 0.5).sum()}")

# These ARE genuinely slow neutrons with Ekin matching eToF.
# The real question is: do we WANT to label them as signal?
# In the reference notebook, what did the original n0_label look like?

# Check the label column (which is the GNN training target, = n0_label)
# versus what might be a "prompt neutron" definition
print(f"\n=== Signal hits Ekin distribution ===")
for lo, hi in [(0, 0.1), (0.1, 0.3), (0.3, 0.5), (0.5, 1.0), (1.0, 2.0), (2.0, 5.0), (5.0, 10.0)]:
    n = ((sig.Ekin >= lo) & (sig.Ekin < hi)).sum()
    print(f"  Ekin [{lo:.1f}, {hi:.1f}): {n} ({n/len(sig)*100:.1f}%)")

# Check the fTime distribution — are these late arrivals?
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# fTime vs eToF for signal
axes[0].hist2d(sig.fTime, sig.eToF, range=[[20,100],[0,5]], bins=80, norm=LogNorm())
axes[0].set_xlabel('fTime [ns]'); axes[0].set_ylabel('eToF [GeV]')
axes[0].set_title('Signal: fTime vs eToF')

# The eToF window for different fTime ranges
if 'eToF_up_2sig' in sig.columns:
    axes[1].hist2d(sig.fTime, sig.etof_window, range=[[20,100],[0,3]], bins=80, norm=LogNorm())
    axes[1].set_xlabel('fTime [ns]'); axes[1].set_ylabel('eToF window width [GeV]')
    axes[1].set_title('Signal: ToF window width vs fTime')

# Ekin vs eToF, zoomed to low energy
axes[2].hist2d(sig.Ekin, sig.eToF, range=[[0,1],[0,1]], bins=80, norm=LogNorm())
axes[2].plot([0,1],[0,1],'r--',lw=1)
axes[2].set_xlabel('Ekin [GeV]'); axes[2].set_ylabel('eToF [GeV]')
axes[2].set_title('Signal: Ekin vs eToF (zoom <1 GeV)')

plt.tight_layout()
plt.show()

print(f"\n=== Side distribution for signal ===")
print(sig.Side.value_counts())

In [ ]:
# ── Check label vs n0_label: are they the same thing? ─────────────────────
# n0_label is from load_hits() → eToF window matching 
# label is the GNN hit target → from graph_dataset _build_single_graph
# They SHOULD be identical. Let's verify.

print("=== label vs n0_label cross-check ===")
print(pd.crosstab(testdf.n0_label, testdf.label, margins=True, 
                  rownames=['n0_label'], colnames=['label (GNN target)']))

# Now the actual question: what is "eToF" in the plot?
# eToF is the time-of-flight energy: eToF = f(distance, fTime)
# For the signal definition (n0_label), eToF is NOT used directly — 
# the cut is: Ekin ∈ [eToF_dn_2sig, eToF_up_2sig] & Side ∈ {5,6}
# So n0_label == 1 means the MC Ekin is CONSISTENT with the measured eToF within 2σ.
# This means by definition eToF ≈ Ekin for signal hits.

# The "peak at low energies" is real slow neutrons. But are they physical?
# Let's check: are these secondary neutrons from nuclear interactions?

sig = testdf[testdf.n0_label == 1].copy()
print(f"\n=== Breakdown of signal by fMotherId ===")
print(f"Primary (fMotherId == -1): {(sig.fMotherId == -1).sum()} ({(sig.fMotherId == -1).mean()*100:.1f}%)")
print(f"Secondary (fMotherId >= 0): {(sig.fMotherId >= 0).sum()} ({(sig.fMotherId >= 0).mean()*100:.1f}%)")

# What PDGs are the parents of secondary signal neutrons?
sec = sig[sig.fMotherId >= 0]
print(f"\nSecondary neutron Ekin < 0.5 GeV: {(sec.Ekin < 0.5).sum()}")
print(f"Primary neutron Ekin < 0.5 GeV: {(sig[sig.fMotherId == -1].Ekin < 0.5).sum()}")

# HYPOTHESIS: the low-energy peak is from secondary neutrons
# produced in nuclear interactions inside the detector material.
# These are labeled as "signal" by the eToF matching cut because they
# are genuinely neutrons whose eToF matches their Ekin.

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compare eToF distributions: primary vs secondary vs all
axes[0].hist(sig[sig.fMotherId == -1].eToF, range=[0, 5], bins=50,
             color='blue', histtype='step', lw=2, label=f'primary n (fMotherId==-1)')
axes[0].hist(sig[sig.fMotherId >= 0].eToF, range=[0, 5], bins=50,
             color='red', histtype='step', lw=2, label=f'secondary n (fMotherId≥0)')
axes[0].axvline(0.5, color='black', ls='--', lw=1, label='0.5 GeV cut')
axes[0].set_xlabel('eToF [GeV]'); axes[0].set_ylabel('hits'); axes[0].set_yscale('log')
axes[0].legend(); axes[0].set_title('eToF: primary vs secondary signal')

# Nprompts distribution for low-E vs high-E neutrons
low_e_ids = sig[sig.Ekin < 0.5][['Row','Id']].drop_duplicates()
high_e_ids = sig[sig.Ekin >= 0.5][['Row','Id']].drop_duplicates()

# How many hits do low-E vs high-E neutron tracks have?
low_e_nhits = sig[sig.Ekin < 0.5].groupby(['Row','Id']).size()
high_e_nhits = sig[sig.Ekin >= 0.5].groupby(['Row','Id']).size()
print(f"\n=== Track multiplicity ===")
print(f"Low-E (<0.5 GeV) neutrons: {len(low_e_nhits)} tracks, mean hits/track = {low_e_nhits.mean():.1f}")
print(f"High-E (≥0.5 GeV) neutrons: {len(high_e_nhits)} tracks, mean hits/track = {high_e_nhits.mean():.1f}")

axes[1].hist(low_e_nhits.values, range=[0, 20], bins=20, color='red', histtype='step', lw=2, 
             label=f'Ekin < 0.5 ({len(low_e_nhits)} tracks)')
axes[1].hist(high_e_nhits.values, range=[0, 20], bins=20, color='blue', histtype='step', lw=2,
             label=f'Ekin ≥ 0.5 ({len(high_e_nhits)} tracks)')
axes[1].set_xlabel('Nhits per track'); axes[1].set_ylabel('tracks'); axes[1].set_yscale('log')
axes[1].legend(); axes[1].set_title('Track size: low-E vs high-E signal neutrons')

# Check: are these single-hit tracks? (Nprompts == 1)
print(f"\n=== Nprompts for low-E signal ===")
low_e_sig = sig[sig.Ekin < 0.5]
print(f"Nprompts == 1: {(low_e_sig.Nprompts == 1).sum()} ({(low_e_sig.Nprompts == 1).mean()*100:.1f}%)")
print(f"Nprompts > 1:  {(low_e_sig.Nprompts > 1).sum()} ({(low_e_sig.Nprompts > 1).mean()*100:.1f}%)")

# Summary: should these be labeled as signal or not?
print(f"\n{'='*60}")
print(f"SUMMARY: The low-eToF peak IS physically correct.")
print(f"These are genuine slow neutrons (Ekin matches eToF).")
print(f"  - {len(low_e_sig)} signal hits with Ekin < 0.5 GeV ({len(low_e_sig)/len(sig)*100:.1f}%)")
print(f"  - Of these, {(low_e_sig.fMotherId >= 0).sum()} are secondary ({(low_e_sig.fMotherId >= 0).mean()*100:.1f}%)")
print(f"  - Of these, {(low_e_sig.Nprompts == 1).sum()} are single-hit tracks ({(low_e_sig.Nprompts == 1).mean()*100:.1f}%)")
print(f"\nIf you want to EXCLUDE slow/secondary neutrons from the signal definition:")
print(f"  Option A: Add Ekin > threshold (e.g., 0.3 or 0.5 GeV)")
print(f"  Option B: Require fMotherId == -1 (primary neutrons only)")
print(f"  Option C: Require Nprompts > 1 (multi-hit tracks only, as in vNn)")

axes[2].hist(sig.Ekin, range=[0, 5], bins=50, color='green', histtype='step', lw=2, label='all signal')
axes[2].hist(sig[sig.fMotherId == -1].Ekin, range=[0, 5], bins=50, color='blue', histtype='step', lw=2, label='primary only')
axes[2].hist(sig[sig.Nprompts > 1].Ekin, range=[0, 5], bins=50, color='orange', histtype='step', lw=2, label='Nprompts > 1')
axes[2].axvline(0.5, color='black', ls='--', lw=1)
axes[2].set_xlabel('Ekin [GeV]'); axes[2].set_ylabel('hits'); axes[2].set_yscale('log')
axes[2].legend(); axes[2].set_title('Signal Ekin: different definitions')